# 2장 모델은 어떻게 만들어지는가 (풀이)

검증용 풀이 노트북입니다. 공유하지 않습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(1장의 공개 LLM, 임베딩 모델)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install transformers torch
# 2장의 예제는 PC에서 공개 소형 LLM 두 개(베이스 모델과 instruct 모델)를 실행합니다. 처음 한 번 각각 약 1GB를 내려받습니다
# API 키가 없어도 예제 2-1, 2-2와 실습은 실행할 수 있습니다. 아래 연결 확인에서 오류가 나도
# "PC에서 실행하는 모델" 줄이 출력됐으면 계속 진행합니다. 키가 없으면 "준비 (1)"의 입력 칸에서 Enter만 누릅니다
import embed
import torch
BASE_MODEL = 'Qwen/Qwen2.5-0.5B'                  # 사전 학습만 마친 베이스 모델
INSTRUCT_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'     # 같은 베이스 모델에 SFT와 강화 학습을 더한 모델
for name in [BASE_MODEL, INSTRUCT_MODEL]:
    embed.ask_endpoint(name)                      # 모델이 PC에 없고 HF_ENDPOINT가 비어 있으면 주소를 묻습니다
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as transformers_logging
transformers_logging.disable_progress_bar()
tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_MODEL)   # 두 모델은 어휘가 같아 토크나이저 하나로 씁니다
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL).eval()
instruct_model = AutoModelForCausalLM.from_pretrained(INSTRUCT_MODEL).eval()
for local_model in [base_model, instruct_model]:  # 두 모델 모두 반복 억제 없이 언제나 1위 토큰을 고르게(greedy decoding) 설정합니다
    local_model.generation_config.update(do_sample=False, temperature=None, top_p=None, top_k=None,
                                         repetition_penalty=None)


def generate(model, characters, start, length=30):
    """예제 2-1용. 앞 8글자를 보고 다음 글자를 샘플링해 붙이는 일을 되풀이합니다(1장의 네 단계)"""
    ids = [characters.index(ch) for ch in start]
    for _ in range(length):
        with torch.no_grad():
            logits = model(torch.tensor([ids[-8:]]))[0]
        ids.append(torch.multinomial(torch.softmax(logits, dim=-1), 1).item())
    return ''.join(characters[i] for i in ids).replace('\n', ' / ')


print('PC에서 실행하는 모델:', BASE_MODEL, INSTRUCT_MODEL)


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 2-1. 무작위 모델을 학습시키기

교재의 절: "사전 학습: 글을 이어 쓰는 법"

글자 하나를 토큰 하나로 쓰는 아주 작은 모델(파라미터 약 11만 개)을 무작위 값에서 시작해 후공정 문장 30개로 1,000번 고칩니다. 학습 전, 100번, 1,000번 고친 뒤에 같은 시작 글("와이어 본딩은 ")로 생성한 글과 손실을 비교합니다. `generate()`는 준비 셀에 있는 함수로, 1장의 네 단계대로 다음 글자를 샘플링해 붙입니다. PC에서 몇 초 안에 끝납니다.

In [ ]:
text = open(f'{DATA}/후공정_문장.txt', encoding='utf-8').read()   # 후공정 문장 30개
characters = sorted(set(text))                                           # 이 예제의 어휘: 글자 하나가 토큰 하나
ids = torch.tensor([characters.index(ch) for ch in text])
inputs = torch.stack([ids[i:i + 8] for i in range(len(ids) - 8)])        # 앞 8글자를 보고
targets = ids[8:]                                                        # 다음 글자를 맞힙니다
torch.manual_seed(0)
tiny_model = torch.nn.Sequential(torch.nn.Embedding(len(characters), 32), torch.nn.Flatten(),
                                 torch.nn.Linear(8 * 32, 256), torch.nn.Tanh(),
                                 torch.nn.Linear(256, len(characters)))   # 파라미터는 무작위 값에서 시작
optimizer = torch.optim.AdamW(tiny_model.parameters(), lr=0.003)        # 손실이 줄어드는 쪽으로 파라미터를 고치는 도구
for step in range(1001):
    batch = torch.randint(len(targets), (64,))                           # 64개씩 묶어
    loss = torch.nn.functional.cross_entropy(tiny_model(inputs[batch]), targets[batch])  # 손실을 계산하고
    optimizer.zero_grad(); loss.backward(); optimizer.step()             # 파라미터를 고칩니다
    if step in (0, 100, 1000):
        print(f'{step:>5}번째 손실 {loss.item():.2f}  생성: {generate(tiny_model, characters, "와이어 본딩은 ")}')

## 예제 2-2. 베이스 모델과 instruct 모델

교재의 절: "사전 학습: 글을 이어 쓰는 법"

같은 질문을 두 모델에 "질문만"과 "채팅 형식"으로 넣고, 어느 조합에서 답하는 글이 나오는지 봅니다. 채팅 형식은 질문을 사용자의 말로 표시하고 모델의 답 차례를 여는 표시를 붙인 글이며, 이 규칙을 채팅 템플릿(chat template)이라고 합니다(3장). 두 모델 모두 언제나 1위 토큰을 고르므로(greedy decoding) 다시 실행해도 같은 글이 나옵니다.

In [ ]:
question = '반도체 후공정에서 다이싱이란 무엇입니까?'
chat_format = tokenizer.apply_chat_template(                  # 채팅 형식의 글을 만듭니다(3장)
    [{'role': 'user', 'content': question}], tokenize=False,
    add_generation_prompt=True)                               # 모델의 답 차례를 여는 표시를 붙입니다
for model_name, model in [('베이스 모델', base_model), ('instruct 모델', instruct_model)]:
    for input_name, text in [('질문만', question), ('채팅 형식', chat_format)]:
        input_ids = tokenizer(text, return_tensors='pt').input_ids
        output = model.generate(input_ids, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
        continuation = tokenizer.decode(output[0, input_ids.shape[1]:],   # 입력 뒤에 새로 생성한 토큰만
                                        skip_special_tokens=True)
        print(f'[{model_name}, {input_name}]', continuation.replace('\n', ' / '), '\n')

## 예제 2-3. 틀린 문장의 손실

교재의 절: "환각이 생기는 이유"

베이스 모델로 맞는 문장, 낱말 하나만 바꾼 틀린 문장, 어순을 섞은 문장의 손실을 계산합니다. 손실이 낮을수록 모델이 그럴듯하다고 본다는 뜻입니다. 평균 손실은 토큰 수가 같은 문장끼리 비교해야 하므로 쌍마다 토큰 수를 같게 맞췄습니다. 틀린 문장의 손실이 더 높은지 봅니다.

In [ ]:
def sentence_loss(text):
    """위치마다 -log(실제 다음 토큰의 확률)을 구해 평균한 값"""
    input_ids = tokenizer(text, return_tensors='pt').input_ids
    with torch.no_grad():
        # labels를 넣으면 위치마다 실제 다음 토큰의 -log p를 구해 평균한 손실을 돌려줍니다
        return base_model(input_ids, labels=input_ids).loss.item()   # .item(): 파이썬 숫자로


sentences = [
    '다이싱은 웨이퍼를 개별 칩으로 자르는 공정입니다.',
    '다이싱은 웨이퍼를 개별 칩으로 붙이는 공정입니다.',
    '와이어 본딩은 칩과 리드를 금속선으로 잇는 공정입니다.',
    '와이어 본딩은 칩과 리드를 금속선으로 자르는 공정입니다.',
    '공정입니다 칩으로 웨이퍼를 다이싱은 개별 자르는.',
]
for sentence in sentences:
    print(f'손실 {sentence_loss(sentence):.2f}  {sentence}')

## 예제 2-4. thinking 토큰

교재의 절: "thinking 모델"

쉬운 질문과 조건이 얽힌 일정 문제(정답 "월-A, 화-D, 수-B, 목-C")를 보내 출력 토큰 수(`reply.output_tokens`), 걸린 시간(`reply.seconds`), 응답 글자 수를 비교합니다. 응답은 짧은데 출력 토큰이 많으면 그만큼 thinking에 쓴 것입니다. `max_tokens=8000`은 thinking이 잘리지 않게 넉넉히 둔 상한입니다. 사내 API의 모델이 thinking 모델이 아니면 두 질문의 차이가 작습니다. 이 셀은 API를 2번 호출합니다.

In [ ]:
questions = {
    '쉬운 질문': '3 더하기 4는 얼마입니까? 숫자만 답해 주세요.',
    '일정 문제': ('로트 A, B, C, D를 월, 화, 수, 목에 하루 한 개씩 테스트합니다. '
              'A는 B보다 먼저 합니다. C는 수요일에 할 수 없습니다. '
              'D는 A 바로 다음 날에 합니다. B는 목요일에 할 수 없습니다. '
              '요일별 로트를 "월-?, 화-?, 수-?, 목-?" 형식으로 한 줄만 답해 주세요.'),
}
for name, question in questions.items():
    reply = llm.ask(question, model='소형', max_tokens=8000)
    print(f'[{name}] 출력 토큰 {reply.output_tokens:,}개, {reply.seconds:.1f}초, 응답 {len(reply.text)}자')
    print('  응답:', reply.text.strip()[:200])

## 실습 2-1. 베이스 모델에게 답하게 하기

교재의 절: "개념 확인 실습"

베이스 모델에 "질문: …" "답: …" 예를 하나 붙이고 새 질문을 넣으면 무엇을 생성할지 먼저 `my_guess`에 적습니다. `few_shot_prompt()`는 이 형식의 글을 "답:"으로 끝나게 만들도록 작성합니다. 실행한 뒤 답 뒤에 무엇을 이어 썼는지 확인하고, 이유를 사전 학습으로 설명하면 끝납니다.

힌트: 예 하나는 `f'질문: {example_question}\n답: {example_answer}'`로 만들고, 줄들을 `'\n'.join()`으로 잇습니다. 빈칸을 채우기 전에 실행하면 `assert` 줄에서 멈춥니다.

In [ ]:
def few_shot_prompt(examples, question):
    """(질문, 답) 예 목록과 새 질문으로 '질문: …' '답: …' 형식의 글을 만듭니다"""
    lines = [f'질문: {example_question}\n답: {example_answer}' for example_question, example_answer in examples]
    lines.append(f'질문: {question}\n답:')
    return '\n'.join(lines)


examples = [('웨이퍼란 무엇입니까?', '반도체 칩을 만드는 얇은 원판입니다.')]
my_guess = '다이싱의 뜻을 한 문장으로 답하고, 다음 질문을 이어 쓸 것이다'
my_explanation = '질문과 답이 번갈아 나오는 글에서는 답 다음에 질문이 오는 이어짐에 높은 확률이 붙는다'
prompt = few_shot_prompt(examples, '반도체 후공정에서 다이싱이란 무엇입니까?')
assert prompt, 'few_shot_prompt()를 먼저 작성합니다'
input_ids = tokenizer(prompt, return_tensors='pt').input_ids
output = base_model.generate(input_ids, max_new_tokens=60, pad_token_id=tokenizer.eos_token_id)
print(prompt + tokenizer.decode(output[0, input_ids.shape[1]:], skip_special_tokens=True))

## 실습 2-2. 그럴듯함과 사실 구분해 보기

교재의 절: "개념 확인 실습"

맞는 문장과 낱말 하나만 바꾼 틀린 문장의 쌍을 세 개 만듭니다. 토큰 수가 달라질 수 있으므로 평균 대신 -log p의 합으로 비교합니다. `total_loss()`는 예제 2-3의 `sentence_loss()`(평균)에 예측한 위치의 수(토큰 수 - 1)를 곱하도록 작성합니다. 어느 쪽의 합이 낮을지 먼저 `my_guess`에 적고, 예측과 다른 쌍의 이유를 `my_explanation`에 적으면 끝납니다.

In [ ]:
def total_loss(text):
    """문장 전체의 -log p 합. 평균 손실에 예측한 위치의 수(토큰 수 - 1)를 곱합니다"""
    token_count = len(tokenizer(text).input_ids)
    return sentence_loss(text) * (token_count - 1)


pairs = [
    ('몰딩은 칩을 수지로 감싸 보호하는 공정입니다.', '몰딩은 칩을 수지로 감싸 가열하는 공정입니다.'),
    ('웨이퍼는 주로 실리콘으로 만듭니다.', '웨이퍼는 주로 유리로 만듭니다.'),
    ('반도체 칩은 전기 신호를 처리합니다.', '반도체 칩은 전기 신호를 차단합니다.'),
]
my_guess = ['맞는 문장', '맞는 문장', '맞는 문장']
my_explanation = '모델은 실제로 이어진 글을 학습했을 뿐이라, 사실이 아닌 낱말도 글로서 자연스러우면 확률이 높다'
for (true_sentence, false_sentence), guess in zip(pairs, my_guess):
    true_total, false_total = total_loss(true_sentence), total_loss(false_sentence)
    lower = '맞는 문장' if true_total < false_total else '틀린 문장'
    print(f'맞는 문장 {true_total:.1f}, 틀린 문장 {false_total:.1f} → 합이 낮은 쪽: {lower} (예측: {guess})')